# Lesson 3.3 — Base-Frame vs Tool-Frame Jacobian
**Module 6 · Unit 3 · Lesson 11**

$J_{\text{tool}}=\mathrm{blkdiag}((R^n_0)^\top,(R^n_0)^\top)\,J_{\text{base}}$ — the rotation-only ($\mathbf d=0$) twist transform from L1.4. We verify it leaves the physical motion invariant.

In [1]:
import numpy as np
def skew(v):
    v=np.asarray(v,float).ravel(); return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
def vee(S): return np.array([S[2,1],S[0,2],S[1,0]])
def dh(th,d,a,al):
    ct,st,ca,sa=np.cos(th),np.sin(th),np.cos(al),np.sin(al)
    return np.array([[ct,-st*ca,st*sa,a*ct],[st,ct*ca,-ct*sa,a*st],[0,sa,ca,d],[0,0,0,1]])
def forward_chain(P,T,q):
    M=np.eye(4); Ms=[M.copy()]
    for i,(th0,d0,a,al) in enumerate(P):
        th,d=(th0+q[i],d0) if T[i]=="R" else (th0,d0+q[i])
        M=M@dh(th,d,a,al); Ms.append(M.copy())
    return Ms
def geometric_jacobian(P,T,q):
    Ms=forward_chain(P,T,q); on=Ms[-1][:3,3]; J=np.zeros((6,len(q)))
    for i in range(len(q)):
        z=Ms[i][:3,2]; o=Ms[i][:3,3]
        if T[i]=="R": J[:3,i]=np.cross(z,on-o); J[3:,i]=z
        else: J[:3,i]=z
    return J
def pose(P,T,q):
    M=forward_chain(P,T,q)[-1]; return M[:3,3], M[:3,:3]

# ZYX roll-pitch-yaw
def rpy_from_R(R):
    return np.array([np.arctan2(R[2,1],R[2,2]),
                     np.arctan2(-R[2,0],np.hypot(R[2,1],R[2,2])),
                     np.arctan2(R[1,0],R[0,0])])   # [roll, pitch, yaw]
def R_from_rpy(rpy):
    r,p,y=rpy
    Rz=np.array([[np.cos(y),-np.sin(y),0],[np.sin(y),np.cos(y),0],[0,0,1.0]])
    Ry=np.array([[np.cos(p),0,np.sin(p)],[0,1,0],[-np.sin(p),0,np.cos(p)]])
    Rx=np.array([[1,0,0],[0,np.cos(r),-np.sin(r)],[0,np.sin(r),np.cos(r)]])
    return Rz@Ry@Rx

# spatial 3R test arm
ARM=([(0,0,0,np.pi/2),(0,0,1.0,0),(0,0,0.8,0)],["R","R","R"])


## Form the tool-frame Jacobian by rotating both blocks

In [2]:
checks=[]
P,T=ARM; q=np.array([0.4,0.6,-0.3])
Jb=geometric_jacobian(P,T,q); p,Rn=pose(P,T,q)
Z=np.zeros((3,3)); Rot=np.block([[Rn.T,Z],[Z,Rn.T]])
Jt=Rot@Jb
print("R^n_0 =\n",np.round(Rn,3))

R^n_0 =
 [[ 0.88  -0.272  0.389]
 [ 0.372 -0.115 -0.921]
 [ 0.296  0.955  0.   ]]


## Invariance: rotating the tool-frame twist back recovers the base-frame twist

In [3]:
qd=np.array([0.7,-1.1,0.5])
xi_base=Jb@qd; xi_tool=Jt@qd
back=np.hstack([Rn@xi_tool[:3], Rn@xi_tool[3:]])
print("base twist :",np.round(xi_base,4))
print("tool->base :",np.round(back,4))
checks.append(np.allclose(back,xi_base,atol=1e-9))

base twist : [ 0.2694  1.322  -1.3664 -0.2337  0.5526  0.7   ]
tool->base : [ 0.2694  1.322  -1.3664 -0.2337  0.5526  0.7   ]


## No lever-arm term: same reference point means rotation only

In [4]:
# the transform has zero off-diagonal blocks (no omega x d coupling)
checks.append(np.allclose(Rot[:3,3:],0) and np.allclose(Rot[3:,:3],0))
print("off-diagonal blocks zero (rotation only):",checks[-1])

off-diagonal blocks zero (rotation only): True


In [5]:
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

All checks passed.
